# League of Legends 👑

**Name(s)**: Hsin Yu Ho

**Website Link**: (your website link)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

# from dsc80_utils import * # Feel free to uncomment and use this.

## Introduction
League of Legends (LoL) is a multiplayer online battle arena (MOBA) game developed by Riot Games. There are more than 150 million players playing this game, which is one of the world’s most popular esports.

The project uses 2022 League of Legends esports match dataset from Oracle’s Elixir. This dataset contains a series of key game statistics and results from League of Legends matches, providing insights into which factors, champion pairings, tactics, or other factors can lead to victory. It includes various features such as banned/ picked champions, in-game statistics, and so on. 

In League of Legends, teams win by destroying the enemy Nexus. To reach the Nexus, teams must first destroy defensive structures such as turrets and inhibitors across the map. Because turrets provide map control and protect important objectives, teams that destroy more turrets often gain significant strategic advantages. This project investigates how objective control, especially turret destruction, relates to winning professional League of Legends matches.


## Research Question
How does the number of turrets destroyed relate to match outcomes in professional League of Legends games?

## Hypothesis
**Null Hypothesis**: Teams with more turret destructions are not more likely to win than teams with more kills. 

**Alternative Hypothesis**: Teams with more turret destructions are more likely to win than teams with more kills. 

**Test Statistic**: The difference between the correlation of towers with result and the correlation of kills with result.

## Data Cleaning and Exploratory Data Analysis
### Data Cleaning

In [ ]:
lol = pd.read_csv('2022_LoL.csv')
lol.shape

In [ ]:
lol_team = lol[lol['position'] == 'team'].copy()
lol_team.head()

Keep only necessary columns:

Check missing data:

In [ ]:
lol_team.isna().sum()

In [ ]:
lol_team.describe()

### Univariate Analysis

In [ ]:
fig_towers = px.histogram(
    lol_team,
    x='towers',
    nbins=12,
    title='Distribution of Towers Destroyed'
)
fig_towers.update_traces(
    marker_line_color='white',
    marker_line_width=0.1
)

In [ ]:
fig_kills = px.histogram(
    lol_team,
    x='teamkills',
    nbins=20,
    title='Distribution of Team Kills'
)
fig_kills.update_traces(
    marker_line_color='white',
    marker_line_width=0.1
)

### Bivariate Analysis
#### Towers vs Win

In [ ]:
lol_team['match_result'] = lol_team['result'].map({0: 'Loss', 1: 'Win'})
fig_tower_win = px.box(
    lol_team,
    x='match_result',
    y='towers',
    title='Towers Destroyed vs Match Result',
    labels={
        'match_result': 'Match Result',
        'towers': 'Towers Destroyed'
    }
)
fig_tower_win.show()

#### Kills vs Win

In [ ]:
fig_kills_win = px.box(
    lol_team,
    x='match_result',
    y='teamkills',
    title='Kills vs Match Result',
    labels={
        'match_result': 'Match Result',
        'kills': 'Kills'
    }
)
fig_kills_win.show()

### Interesting Aggregates
#### Pivot Table

In [ ]:
pivot = pd.pivot_table(
    lol_team,
    values=['teamkills', 'towers'],
    index='match_result',
    aggfunc=np.mean
)
pivot

#### Correlation Table & Heatmap

In [ ]:
corr = lol_team[['result', 'teamkills', 'towers']].corr()
corr

In [ ]:
px.imshow(
    corr,
    text_auto=True,
    title='Correlation Heatmap'
)


## Step 3: Assessment of Missingness

## NMAR Analysis
The missingness of the `url` column does not depend on the league column which is a NMAR (Not Mising at Random). Because this missingness may depend on whether a match webpage or official record actually existed when the data was collected. 

## Missingness Dependency
The column `assistsat25` depends on `datacompleteness` because `datacompleteness` rows labeled as partial are much more likely to have missing values for `assistsat25`, while rows labeled as complete usually contain valid values. 

### Permutation Test: Missingness of assistsat25 vs datacompleteness

- **Null Hypothesis**: The missingness of `assistsat25` does not depend on `datacompleteness`.

- **Alternative Hypothesis**: The missingness of `assistsat25` does depend on `datacompleteness`.

In [ ]:
lol['assists_nan'] = lol['assistsat25'].isna()

In [ ]:
obs = (
    lol.pivot_table(
        index='assists_nan',
        columns='datacompleteness',
        aggfunc='size',
        observed=False
    )
    .apply(lambda x: x / x.sum(), axis=1)
)

obs_tvd = (obs.iloc[0] - obs.iloc[1]).abs().sum() / 2
obs_tvd

In [ ]:
tvds = []

for _ in range(500):
    shuffled = np.random.permutation(lol['assists_nan'])
    shuffled_df = (
        lol.assign(shuffled=shuffled)
        .pivot_table(
            index='shuffled',
            columns='datacompleteness',
            aggfunc='size',
            observed=False
        )
        .apply(lambda x: x / x.sum(), axis=1)
    )

    tvd = (shuffled_df.iloc[0] - shuffled_df.iloc[1]).abs().sum() / 2
    tvds.append(tvd)

p_value = np.mean(np.array(tvds) >= obs_tvd)
p_value

In [ ]:
fig = px.histogram(
    x=tvds,
    nbins=30,
    title='Permutation Distribution of TVD (Missingness assistsat25 vs DataComplete)',
    labels={'x': 'TVD'}
)
fig.add_vline(
    x=obs_tvd,
    line_color='red',
    annotation_text='Observed TVD'
)
fig.update_traces(
    marker_line_color='white',
    marker_line_width=0.1
)
fig.show()

The p-value from the permutation test is extremely small and close to 0. This indicates that the observed TVD is very unlikely to occur under the null hypothesis. Therefore, there is strong evidence that the missingness of assistsat25 depends on datacompleteness, we reject null hypothesis.

### Permutation Test: Missingness of assistsat25 vs datacompleteness
The variable `assistsat25` does not depend on `side` because whether a team is blue side or red side should NOT affect whether `assistsat25` is missing.
- **Null Hypothesis**: The missingness of `assistsat25` does not depend on `side`.

- **Alternative Hypothesis**: The missingness of `assistsat25` does depend on `side`.

In [ ]:
obs = (
    lol.pivot_table(
        index='assists_nan',
        columns='side',
        aggfunc='size',
        observed=False
    )
    .apply(lambda x: x / x.sum(), axis=1)
)

obs_tvd = (obs.iloc[0] - obs.iloc[1]).abs().sum() / 2
obs_tvd

In [ ]:
tvds = []

for _ in range(500):
    shuffled = np.random.permutation(lol['assists_nan'])
    shuffled_df = (
        lol.assign(shuffled=shuffled)
        .pivot_table(
            index='shuffled',
            columns='side',
            aggfunc='size',
            observed=False
        )
        .apply(lambda x: x / x.sum(), axis=1)
    )

    tvd = (shuffled_df.iloc[0] - shuffled_df.iloc[1]).abs().sum() / 2
    tvds.append(tvd)

p_value = np.mean(np.array(tvds) >= obs_tvd)
p_value

In [ ]:
fig = px.histogram(
    x=tvds,
    nbins=30,
    title='Permutation Distribution of TVD (Missingness assistsat25 vs Side)',
    labels={'x': 'TVD'},
    
)

fig.add_vline(
    x=obs_tvd,
    line_color='red',
    annotation_text='Observed TVD'
)

fig.update_traces(
    marker_line_color='white',
    marker_line_width=0.1
)
fig.show()

## Step 4: Hypothesis Testing
**Null Hypothesis**: Teams with more turret destructions are not more likely to win than teams with more kills. 

**Alternative Hypothesis**: Teams with more turret destructions are more likely to win than teams with more kills. 

I will use **permutation test** and **p-value** to see if turret destructions are more likely to win than teams with more kills or not.

In [ ]:
lol_team['more_towers'] = lol_team['towers'] > lol_team['opp_towers']
lol_team['more_kills'] = lol_team['teamkills'] > lol_team['teamdeaths']

# Win rate with more towers
tower_win = lol_team.loc[lol_team['more_towers'], 'result'].mean()
# Win rate with more teamkills
kill_win = lol_team.loc[lol_team['more_kills'], 'result'].mean()
observed = tower_win - kill_win
observed

In [ ]:
diffs = []

for _ in range(500):
    shuffle = np.random.permutation(lol_team['result'])
    shuffle_df = lol_team.assign(shuffled_result=shuffle)
    tower_rate = shuffle_df.loc[shuffle_df['more_towers'], 'shuffled_result'].mean()
    kill_rate = shuffle_df.loc[shuffle_df['more_kills'], 'shuffled_result'].mean()
    diffs.append(tower_rate - kill_rate)

p_value = np.mean(np.array(diffs) >= observed)
p_value

In [ ]:
fig = px.histogram(
    x=diffs,
    nbins=30,
    title='Permutation Distribution of Difference between Mean Towers and Mean Kills',
    labels={'x': 'Difference in Mean Towers and Mean Kills'}
)

fig.add_vline(
    x=observed,
    line_color='red',
    annotation_text='Observed Statistic'
)

fig.update_traces(
    marker_line_color='white',
    marker_line_width=0.1
)

fig.show()

Since the p-value is less than 0.05, I reject the null hypothesis. This provides strong evidence that teams with more destroyed towers are more likely to win than teams with more kills.

## Step 5: Problem Identification

The prediction problem for this project is to predict whether a team wins or loses a professional League of Legends match based on in-game team statistics (towers destroyed/ kills).

The response variable I used is `result`, has two possible outcomes is a **binary classification problem**:

* `1` = win
* `0` = loss

Potential predictor variables include team statistics such as `towers`, `teamkills`, `dragons`, `barons`, `golddiffat25`, `xpdiffat25`, and `csdiffat25`.

This prediction problem is to understand which in-game statistics are most predictive of winning, which can help identify the factors most strongly associated with success in professional League of Legends matches. It also aligns with the overall theme of this project, which investigates whether tower destruction is more strongly related to winning than kills.

## Step 6: Baseline Model
For my baseline model, I used **Logistic Regression** with three features: `towers`, `teamkills`, and `side`. The variables `towers` and `teamkills` are numerical and were left unchanged, while the categorical variable `side` was transformed using **One-Hot Encoding**. All preprocessing and model training steps were implemented within a single sklearn Pipeline.

I evaluated the model using a train-test split to assess its ability to generalize to unseen matches. Logistic Regression was chosen because it is a simple and interpretable classification model commonly used as a baseline for binary prediction tasks.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


X = lol_team[['towers', 'teamkills', 'side']]
y = lol_team['result']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

preproc = ColumnTransformer(
    [
        ('onehot', OneHotEncoder(drop='if_binary'), ['side'])
    ],
    remainder='passthrough'
)

baseline_model = Pipeline([
    ('preproc', preproc),
    ('clf', LogisticRegression(max_iter=1000))
])

baseline_model.fit(X_train, y_train)

train_acc = baseline_model.score(X_train, y_train)
test_acc = baseline_model.score(X_test, y_test)

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

## Step 7: Final Model

In [ ]:
# TODO

## Step 8: Fairness Analysis

In [ ]:
# TODO